#### Libraries

In [35]:
import pandas as pd
import matplotlib.pyplot as plt
import re
import folium
from folium.plugins import MarkerCluster
import matplotlib.dates as mdates

In [2]:
df = pd.read_csv('nyc_bus_cleaned.csv')

In [3]:
df.head()

,RecordedAtTime,DirectionRef,PublishedLineName,OriginName,OriginLat,OriginLong,DestinationName,DestinationLat,DestinationLong,VehicleRef,VehicleLocation.Latitude,VehicleLocation.Longitude,NextStopPointName,ArrivalProximityText,DistanceFromStop,ExpectedArrivalTime,ScheduledArrivalTime,Scheduled_dt,delay_minutes
0,2017-06-01 00:03:34,0.0,B8,4 AV/95 ST,40.616104,-74.031143,BROWNSVILLE ROCKAWAY AV,40.656048,-73.907379,NYCT_430,40.635170,-73.960803,FOSTER AV/E 18 ST,approaching,76.0,2017-06-01 00:03:59,00:06:14,2017-06-01 00:06:14,-2.25
1,2017-06-01 00:03:43,1.0,S61,ST GEORGE FERRY/S61 & S91,40.643169,-74.073494,S I MALL YUKON AV,40.575935,-74.167686,NYCT_8263,40.590802,-74.158340,MERRYMOUNT ST/TRAVIS AV,approaching,62.0,2017-06-01 00:03:56,23:58:02,2017-05-31 23:58:02,5.90
2,2017-06-01 00:03:49,0.0,Bx10,E 206 ST/BAINBRIDGE AV,40.875008,-73.880142,RIVERDALE 263 ST,40.912376,-73.902534,NYCT_4223,40.886010,-73.912647,HENRY HUDSON PKY E/W 235 ST,at stop,5.0,2017-06-01 00:03:56,00:00:53,2017-06-01 00:00:53,3.05
3,2017-06-01 00:03:31,0.0,Q5,TEARDROP/LAYOVER,40.701748,-73.802399,ROSEDALE LIRR STA via MERRICK,40.666012,-73.735939,NYCT_8422,40.668002,-73.729348,HOOK CREEK BL/SUNRISE HY,< 1 stop away,267.0,2017-06-01 00:04:03,00:03:00,2017-06-01 00:03:00,1.05
4,2017-06-01 00:03:22,1.0,Bx1,RIVERDALE AV/W 231 ST,40.881187,-73.909340,MOTT HAVEN 136 ST via CONCOURSE,40.809654,-73.928360,NYCT_4710,40.868134,-73.893032,GRAND CONCOURSE/E 196 ST,at stop,11.0,2017-06-01 00:03:56,23:59:38,2017-05-31 23:59:38,4.30


In [8]:
# Datetime Object Conversion
df['RecordedAtTime'] = pd.to_datetime(df['RecordedAtTime'])
df['ExpectedArrivalTime'] = pd.to_datetime(df['ExpectedArrivalTime'])
df['Scheduled_dt'] = pd.to_datetime(df['Scheduled_dt'])

#### Finding Bus Routes and their Stops

In [9]:
# ── Find all unique bus routes ─────────────────────────────────────────────
route_counts = df['PublishedLineName'].value_counts()

print(f"Total unique routes: {df['PublishedLineName'].nunique()}")
print(f"\nTop 20 routes by number of observations:")
print(route_counts.head(20))

Total unique routes: 242

Top 20 routes by number of observations:
PublishedLineName
B6          390695
B41         348053
Q58         314879
Q44-SBS     295141
B35         294302
Bx36        288273
M15-SBS     272554
M101        264437
B82         260572
Q27         259617
Bx15        253348
B15         250589
B46         246952
M4          222675
Bx19        221384
B38         219917
M15         215755
Bx12-SBS    213621
Bx9         211895
B46-SBS     197593
Name: count, dtype: int64


In [10]:
# Top 5 routes by number of observations
top_5_routes = df['PublishedLineName'].value_counts().head(5).index.tolist()

print("Top 5 routes by number of observations:")
print(top_5_routes)

for route_name in top_5_routes:
    stops = (
        df[df['PublishedLineName'] == route_name]['NextStopPointName']
        .value_counts()
        .reset_index()
    )
    stops.columns = ['StopName', 'Observations']
    
    print(f"\nStops for route {route_name} ({len(stops)} unique stops):")
    print(stops)

Top 5 routes by number of observations:
['B6', 'B41', 'Q58', 'Q44-SBS', 'B35']

Stops for route B6 (119 unique stops):
                     StopName  Observations
0     GLENWOOD RD/NOSTRAND AV         15736
1        AV J/CONEY ISLAND AV         13716
2               AV H/UTICA AV         13113
3         HARWAY AV/BAY 37 ST         11949
4      ASHFORD ST/NEW LOTS AV         10886
..                        ...           ...
114    GLENWOOD RD/BEDFORD AV           694
115       GLENWOOD RD/E 34 ST           647
116      GLENWOOD RD/E 100 ST           485
117      FLATLANDS AV/E 76 ST           380
118  ROCKAWAY PKY/GLENWOOD RD           179

[119 rows x 2 columns]

Stops for route B41 (85 unique stops):
                   StopName  Observations
0     FLATBUSH AV/CHURCH AV         20504
1   FLATBUSH AV/NOSTRAND AV         18635
2   FLATBUSH AV/ATLANTIC AV         17681
3    TILLARY ST/CADMAN PZ W         17330
4     FLATBUSH AV/FOSTER AV         16220
..                      ...          

In [13]:
# Filter dataset
df_top5 = df[df['PublishedLineName'].isin(top_5_routes)].copy()

In [14]:
# Unique origins and destinations for each line
line_od_summary = (
    df_top5.groupby('PublishedLineName')
      .agg(
          n_unique_origins=('OriginName', 'nunique'),
          n_unique_destinations=('DestinationName', 'nunique'),
          origins=('OriginName', lambda x: sorted(pd.Series(x.dropna().unique()).tolist())),
          destinations=('DestinationName', lambda x: sorted(pd.Series(x.dropna().unique()).tolist()))
      )
      .reset_index()
      .sort_values('PublishedLineName')
)

line_od_summary.head()

,PublishedLineName,n_unique_origins,n_unique_destinations,origins,destinations
0,B35,4,5,"[39 ST/1 AV, CHURCH AV/KINGS HY, CHURCH AV/MC ...","[BROWNSVILLE M GASTON BL via CHURCH, LTD BROWN..."
1,B41,9,10,"[CADMAN PLAZA WEST/JOHNSON ST, E 68 ST/AV N, E...","[BERGEN BCH VETERANS AV via FLATBSH, Church AV..."
2,B6,9,6,"[AV J/CONEY ISLAND AV, BAY PY/60 ST, BEDFORD A...","[AVENUE J CONEY IS AV, BENSONHURST HARWAY AV, ..."
3,Q44-SBS,8,4,"[BOSTON RD/E 180 ST, LAFAYETTE AV/HUTCHINSON R...","[SELECT BUS BRONX ZOO via MAIN ST, SELECT BUS ..."
4,Q58,5,5,"[41 RD/MAIN ST, 71 ST/GRAND AV, FRESH POND RD/...","[108 ST H HRDNG EXY, FLUSHING MAIN ST, LTD FLU..."


In [15]:
# Find unique Origin-Destination pairs
od_pairs = df_top5.groupby(['OriginName', 'DestinationName', 'DirectionRef']).size().reset_index(name='Observations')
print("Origin-Destination pairs:")
print(od_pairs)

Origin-Destination pairs:
                           OriginName                         DestinationName  \
0                          39 ST/1 AV      BROWNSVILLE M GASTON BL via CHURCH   
1                          39 ST/1 AV  LTD BROWNSVILLE M GASTON BL via CHURCH   
2                       41 RD/MAIN ST                      LTD RIDGEWOOD TERM   
3                       41 RD/MAIN ST                          RIDGEWOOD TERM   
4                      71 ST/GRAND AV                      108 ST H HRDNG EXY   
..                                ...                                     ...   
59                   PARSONS BL/14 AV          SELECT BUS JAMAICA via MAIN ST   
60  ROCKAWAY STATION/ROCKAWAY STATION                    AVENUE J CONEY IS AV   
61  ROCKAWAY STATION/ROCKAWAY STATION                   BENSONHURST HARWAY AV   
62  ROCKAWAY STATION/ROCKAWAY STATION               LTD BENSONHURST HARWAY AV   
63              STUART ST/FILLMORE AV                              NOSTRND AV   

 

In [16]:
proximity_mask = df_top5['ArrivalProximityText'].isin(['at stop', 'approaching'])
df5c = df_top5[proximity_mask].copy()

def normalize_stop_name(s):
    if pd.isna(s):
        return s
    s = str(s).upper().strip()
    s = ' '.join(s.split())
    return s

df5c['stop_name_clean'] = df5c['NextStopPointName'].apply(normalize_stop_name)

In [22]:
# Global stop table across all 5 routes
stops_master = (
    df5c.groupby('stop_name_clean')
        .agg(
            Latitude=('VehicleLocation.Latitude', 'median'),
            Longitude=('VehicleLocation.Longitude', 'median'),
            Observations=('VehicleLocation.Latitude', 'count'),
            Routes=('PublishedLineName', lambda x: sorted(set(x))),
            RouteCount=('PublishedLineName', 'nunique')
        )
        .reset_index()
        .rename(columns={'stop_name_clean': 'stop_key'})
)

# Optional confidence filter
stops_master = stops_master[stops_master['Observations'] >= 20].copy()

# Attach shared coordinates back to rows
df5c = df5c.merge(
    stops_master[['stop_key', 'Latitude', 'Longitude']],
    left_on='stop_name_clean',
    right_on='stop_key',
    how='left',
    suffixes=('', '_canonical')
)

# Route-stop relationship table
route_stops = (
    df5c.groupby(['PublishedLineName', 'DirectionRef', 'stop_name_clean'])
        .agg(
            stop_lat=('VehicleLocation.Latitude', 'median'),
            stop_lon=('VehicleLocation.Longitude', 'median'),
            obs=('VehicleLocation.Latitude', 'count')
        )
        .reset_index()
        .rename(columns={'stop_name_clean': 'stop_key'})
)

In [31]:
route_stops

,PublishedLineName,DirectionRef,stop_key,stop_lat,stop_lon,obs
0,B35,0.0,14 AV/36 ST,40.641006,-73.982306,925
1,B35,0.0,14 AV/39 ST,40.639481,-73.984300,1408
2,B35,0.0,14 AV/CHURCH AV,40.641736,-73.981549,1037
3,B35,0.0,39 ST/10 AV,40.644695,-73.993030,784
4,B35,0.0,39 ST/12 AV,40.642054,-73.988660,997
...,...,...,...,...,...,...
535,Q58,1.0,PUTNAM AV/ONDERDONK AV,40.702544,-73.904393,1063
536,Q58,1.0,PUTNAM AV/WOODWARD AV,40.703646,-73.903271,907
537,Q58,1.0,PUTNAM AV/WYCKOFF AV,40.698542,-73.908481,1134
538,Q58,1.0,RIDGEWOOD PL/MADISON ST,40.697660,-73.909478,856


In [32]:
stops_unique = (
    route_stops.groupby('stop_key', as_index=False)
    .agg(
        stop_lat=('stop_lat', 'median'),
        stop_lon=('stop_lon', 'median'),
        total_obs=('obs', 'sum')
    )
)

stops_unique.to_csv("all_stop_coordinates.csv", index=False)

In [36]:
# Load stop coordinates
stops = pd.read_csv("all_stop_coordinates.csv")

# Center map on the median stop location
center_lat = stops["stop_lat"].median()
center_lon = stops["stop_lon"].median()

m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=11,
    tiles="CartoDB positron"
)

# Cluster markers so the map stays usable
marker_cluster = MarkerCluster().add_to(m)

for _, row in stops.iterrows():
    folium.CircleMarker(
        location=[row["stop_lat"], row["stop_lon"]],
        radius=4,
        color="blue",
        fill=True,
        fill_opacity=0.6,
        popup=(
            f"<b>Stop:</b> {row['stop_key']}<br>"
            f"<b>Lat:</b> {row['stop_lat']:.6f}<br>"
            f"<b>Lon:</b> {row['stop_lon']:.6f}<br>"
            f"<b>Total observations:</b> {int(row['total_obs'])}"
        ),
        tooltip=row["stop_key"]
    ).add_to(marker_cluster)

m.save("all_stops_map.html")
m